In [ ]:
%matplotlib inline

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch

import cute_rmsnorm
import triton_rmsnorm
import tilelang_rmsnorm
from common.asm import dump_asm
from common.inputs import make_inputs
from common.run import make_runner, run_benchmark
from neuroslop.visualize_bench_results import plot_benchmark_sweep

SHAPES = [(512, 1024), (4096, 3072), (4096, 4096)]  # cute_vec requires widths divisible by 4
EPS = cute_rmsnorm.EPS  # the CUDA kernel hardcodes the same value
BLOCK_SIZE = 512

def rmsnorm(x, w):
    return torch.nn.functional.rms_norm(x, (x.shape[-1],), w, EPS)

In [ ]:
AUTOTUNE = True  # False runs each kernel with its own defaults

TRITON_KERNEL = triton_rmsnorm.rmsnorm_kernel(autotune=AUTOTUNE)

IMPLEMENTATIONS = {
    "torch": make_runner("torch", rmsnorm),
    "torch_compile": make_runner("torch_compile", lambda x, w: x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + EPS) * w),
    "triton": make_runner(
        "triton",
        TRITON_KERNEL,
        extra_args=lambda x, w, out: (x.stride(0), x.shape[1], EPS),
        grid=lambda meta: (meta["x_ptr"].shape[0],),
    ),
    "tilelang": make_runner(
        "tilelang", tilelang_rmsnorm.rmsnorm_,
        specialize=lambda kernel, x, w: kernel(*x.shape, BLOCK_SIZE=BLOCK_SIZE, dtype=str(x.dtype).removeprefix("torch.")),
        name="rmsnorm",
    ),
    "cute": make_runner("cute", cute_rmsnorm.rmsnorm(autotune=AUTOTUNE)),
    "cute_vec": make_runner("cute", cute_rmsnorm.rmsnorm_vec(autotune=AUTOTUNE)),
    "cuda": make_runner("cuda", "cuda_rmsnorm.cu", entry="rmsnorm"),
}

In [ ]:
# Tests: row lengths around block and vector boundaries. cute_vec needs N % 4 == 0.
for m, n in [(1, 1), (3, 7), (5, 255), (2, 256), (3, 257), (7, 1000), (4, 4096), (3, 8196)]:
    inputs = make_inputs((m, n), (n,))
    for name, fn in IMPLEMENTATIONS.items():
        if name == "cute_vec" and n % 4:
            continue
        torch.testing.assert_close(fn(*inputs), rmsnorm(*inputs), msg=f"{name}, {m}x{n}")

# Test the public TileLang wrapper, including FP32 accumulation for low precision.
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    for n in (1, 257, 4096, 8196):
        x = torch.randn(3, n, device="cuda", dtype=dtype)
        w = torch.randn(n, device="cuda", dtype=dtype)
        expected = (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + EPS) * w.float()).to(dtype)
        torch.testing.assert_close(tilelang_rmsnorm.rmsnorm(x, w), expected)
    x = torch.zeros(2, 257, device="cuda", dtype=dtype)
    torch.testing.assert_close(tilelang_rmsnorm.rmsnorm(x, w[:257].contiguous()), x)
x = torch.empty(0, 257, device="cuda")
assert tilelang_rmsnorm.rmsnorm(x, torch.ones(257, device="cuda")).shape == x.shape
print("OK")

In [ ]:
results = run_benchmark(
    IMPLEMENTATIONS,
    inputs=lambda shape: make_inputs(shape, (shape[1],)),
    reference=rmsnorm,
    flops=lambda shape: 4 * shape[0] * shape[1],  # square, add, normalize, scale
    shapes=SHAPES,
)
plot_benchmark_sweep(results, baseline="torch_compile", title=torch.cuda.get_device_name())
plt.show()
if AUTOTUNE:
    print("Autotune configs for final shape", SHAPES[-1])
    print("triton config:", TRITON_KERNEL.best_config)
    print("cute config:", IMPLEMENTATIONS["cute"].asm()["rmsnorm"]["config"])
    print("cute_vec config:", IMPLEMENTATIONS["cute_vec"].asm()["rmsnorm_vec"]["config"])

In [ ]:
# Export assembly after the sweep, using the existing backend directories.
for name, fn in IMPLEMENTATIONS.items():
    asm = fn.asm()
    dump_asm(asm, Path("asm") / name)
    print(name, list(asm))

In [ ]:
# Export the generated CUDA source for the final shape's TileLang kernel.
cuda_dir = Path('cuda') / 'tilelang'
cuda_dir.mkdir(parents=True, exist_ok=True)
source = tilelang_rmsnorm.rmsnorm_(*SHAPES[-1], BLOCK_SIZE=BLOCK_SIZE).get_kernel_source()
(cuda_dir / 'rmsnorm.cu').write_text(source, encoding='utf-8')